# Analisi assi: scala logaritmica e minimo diverso da 0

Dataset: `data/groundtruth/arXiv` e `data/groundtruth/PMCharts`.

**Analisi 1** — Grafici con almeno un asse (x o y) in scala logaritmica (`is_log=True`).

**Analisi 2** — Grafici il cui asse (x o y) ha un minimo esplicitamente annotato e diverso da 0 (`min != None` e `min != 0`).

In [1]:
import json
from pathlib import Path
import pandas as pd

BASE = Path('../data/groundtruth')
DATASETS = ['arXiv', 'PMCharts']

records = []
for dataset in DATASETS:
    for path in sorted((BASE / dataset).glob('**/*.json')):
        chart_type = path.parent.name
        d = json.loads(path.read_text())
        x = d.get('x_axis') or {}
        y = d.get('y_axis') or {}

        has_log = x.get('is_log', False) or y.get('is_log', False)

        x_min, y_min = x.get('min'), y.get('min')
        nonzero_min = (
            (x_min is not None and x_min != 0) or
            (y_min is not None and y_min != 0)
        )

        records.append({
            'dataset': dataset,
            'chart_type': chart_type,
            'file': path.name,
            'has_log': has_log,
            'nonzero_min': nonzero_min,
        })

df = pd.DataFrame(records)
print(f'Grafici totali caricati: {len(df)}')

Grafici totali caricati: 950


---
## Analisi 1 — Grafici con asse logaritmico (x o y)

In [2]:
def summary_table(df, flag_col, label):
    """Builds per-class + global summary for a boolean flag column."""
    agg = (
        df.groupby('chart_type')
        .agg(totale=('file', 'count'), n=(flag_col, 'sum'))
        .reset_index()
    )
    # global row
    total_row = pd.DataFrame([{
        'chart_type': 'TOTALE',
        'totale': agg['totale'].sum(),
        'n': agg['n'].sum(),
    }])
    agg = pd.concat([agg, total_row], ignore_index=True)
    agg['percentuale (%)'] = (100 * agg['n'] / agg['totale']).round(1)
    agg = agg.rename(columns={
        'chart_type': 'Classe',
        'totale': 'Totale',
        'n': label,
    })
    return agg.set_index('Classe')


print('=== Globale + per classe ===\n')
display(summary_table(df, 'has_log', 'N asse log'))

=== Globale + per classe ===



,Totale,N asse log,percentuale (%)
Classe,,,
bar,100,5,5.0
box,100,3,3.0
bubble,50,0,0.0
errorpoint,100,2,2.0
heatmap,100,0,0.0
histogram,100,5,5.0
line,100,6,6.0
pie,100,0,0.0
radar,100,0,0.0


In [3]:
print('=== Per dataset ===\n')
for ds, grp in df.groupby('dataset'):
    print(f'--- {ds} ---')
    display(summary_table(grp, 'has_log', 'N asse log'))
    print()

=== Per dataset ===

--- PMCharts ---


,Totale,N asse log,percentuale (%)
Classe,,,
bar,50,0,0.0
box,50,0,0.0
bubble,50,0,0.0
errorpoint,50,1,2.0
heatmap,50,0,0.0
histogram,50,1,2.0
line,50,1,2.0
pie,50,0,0.0
radar,50,0,0.0



--- arXiv ---


,Totale,N asse log,percentuale (%)
Classe,,,
bar,50,5,10.0
box,50,3,6.0
errorpoint,50,1,2.0
heatmap,50,0,0.0
histogram,50,4,8.0
line,50,5,10.0
pie,50,0,0.0
radar,50,0,0.0
scatter,50,8,16.0


---
## Analisi 2 — Grafici il cui minimo (x o y) non inizia da 0

Condizione: almeno un asse ha `min` esplicitamente annotato, non-nullo e diverso da 0.

In [4]:
print('=== Globale + per classe ===\n')
display(summary_table(df, 'nonzero_min', 'N min≠0'))

=== Globale + per classe ===



,Totale,N min≠0,percentuale (%)
Classe,,,
bar,100,14,14.0
box,100,39,39.0
bubble,50,32,64.0
errorpoint,100,51,51.0
heatmap,100,0,0.0
histogram,100,47,47.0
line,100,41,41.0
pie,100,0,0.0
radar,100,25,25.0


In [5]:
print('=== Per dataset ===\n')
for ds, grp in df.groupby('dataset'):
    print(f'--- {ds} ---')
    display(summary_table(grp, 'nonzero_min', 'N min≠0'))
    print()

=== Per dataset ===

--- PMCharts ---


,Totale,N min≠0,percentuale (%)
Classe,,,
bar,50,5,10.0
box,50,21,42.0
bubble,50,32,64.0
errorpoint,50,23,46.0
heatmap,50,0,0.0
histogram,50,26,52.0
line,50,22,44.0
pie,50,0,0.0
radar,50,17,34.0



--- arXiv ---


,Totale,N min≠0,percentuale (%)
Classe,,,
bar,50,9,18.0
box,50,18,36.0
errorpoint,50,28,56.0
heatmap,50,0,0.0
histogram,50,21,42.0
line,50,19,38.0
pie,50,0,0.0
radar,50,8,16.0
scatter,50,23,46.0
